# HiSparse 入门 (3/8)：Resolver 解算器 —— 把逻辑 token 翻译成物理行

> **一句话结论**：resolver 每步、每个 indexer 组调用一次，完成两件事：把 top-k 逻辑 token 翻译成可直读的物理行号，并产出本步搬移计划（缺哪些行、淘汰哪些槽）。

## 1. 它在流程里的位置

可以把 resolver 想成前台服务员：

- 手里拿着读者要找的书单（逻辑索引）；
- 查书架和仓库（Host / Resident / Hot 页表）；
- 告诉助手去仓库补哪些书（搬移计划），并把桌面上书的位置报给读者（注意力索引）。

## 2. 算法分五步（语义在 A5/A3 两端一致）

### 2.1 fast path（快速通道）

如果请求"完全驻留"：Resident 页表非空、几何满足、没有 hot 需求，直接把逻辑映射到已驻留行，输出物理索引。**不碰 hash、不碰 dgi、不碰 LRU。**

### 2.2 hash union（未命中行合并）

对走不到快速通道的行：从 Host 池取行号，开放寻址插入 hash 表。同一个 Host 行只保留**最小首次引用位置**，这个顺序后面决定谁先被淘汰。

### 2.3 hot 命中

用 dgi 热区槽表与 Host 行号比对：命中就输出 `kHit | slot`（命中标记 + 槽位）；否则记为 miss。

### 2.4 稳定 LRU 分区

- 命中项保持原 LRU 顺序排在前部；
- 未命中项按 LRU 逆序排到尾部（尾部最先被逐出）；
- **stable**：相同输入永远产生相同输出，便于逐位验收。

### 2.5 victim 回填

按首次引用顺序，把 union 里的新行分配到尾部槽：写 `swap_host` / `swap_device` 搬移清单，置 `miss_mask`；最后压缩出注意力索引，空位置填 `-1`，并给出有效长度 counts。

## 3. A5 上的实现形态

- 内核是 **AscendC SIMT AIV**（`csrc/hisparse/hisparse_resolver.cpp`）；
- 每个连续请求组由一个 AIV 一次 launch 完成；
- UB（统一缓冲）预算由环境变量 `HISPARSE_RESOLVER_UB_BYTES=175616` 控制；
- 编译目标面向 dav-3510；A5 上没有 Triton resolver 回退路径。

## 4. 小结

- resolver = 翻译（逻辑→物理）+ 调度（搬移/淘汰计划）；
- 五步语义：fast path → hash union → hot 命中 → 稳定 LRU → victim 回填；
- A5 用 SIMT AIV 内核跑这套算法；
- 下一篇看配套的缓存算子家族。

**下一篇**：`04-cache-ops-kernels.ipynb`
